# Figure 9.12: λ Changes Model Behavior

As λ increases, the fit gets smoother and the polynomial coefficients shrink.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def draw(lam=4.0, degree=8, noise=0.25, seed=9):
    x, y, _, _ = poly_data(seed=seed, n=60, obs_noise=noise, gt_noise=0.0)
    th_ols = ridge_fit(x, y, degree, 0.0)
    th_ridge = ridge_fit(x, y, degree, lam)
    xg = np.linspace(-3, 3, 300)
    fig, axs = plt.subplots(1, 2, figsize=(13.0, 4.4))
    axs[0].scatter(x, y, color="#334155", s=18, label="observed")
    axs[0].plot(xg, base_curve(xg), "--", color="#64748b", lw=2, label="true trend")
    axs[0].plot(xg, poly_predict(th_ols, xg), color="#dc2626", lw=2.5, label="OLS")
    axs[0].plot(xg, poly_predict(th_ridge, xg), color="#2563eb", lw=2.5, label="ridge")
    axs[0].set_title("Model behavior")
    axs[0].legend()
    axs[0].grid(alpha=0.2)
    idx = np.arange(degree + 1)
    axs[1].bar(idx - 0.18, th_ols, width=0.36, color="#fca5a5", label="OLS")
    axs[1].bar(idx + 0.18, th_ridge, width=0.36, color="#93c5fd", label="ridge")
    axs[1].set_title("Coefficient magnitudes")
    axs[1].set_xlabel("coefficient index")
    axs[1].legend()
    axs[1].grid(alpha=0.2)
    plt.show()

widgets.interact(
    draw,
    lam=widgets.FloatSlider(min=0.0, max=40.0, step=0.5, value=4.0, continuous_update=False),
    degree=widgets.IntSlider(min=1, max=12, step=1, value=8, continuous_update=False),
    noise=widgets.FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25, continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=9, continuous_update=False),
)
